# Sesión 4 — Limpiar y ordenar un corpus en lote (Python)

Cualquiera que haya laburado con fuentes lo sabe: bajás recortes de prensa, te pasan un lote de transcripciones, copiás notas de una web vieja… y lo que te queda es un desastre. Archivos con nombres tipo `Nota 1 (copia).txt`, espacios de más, líneas en blanco que sobran, y restos de HTML que se colaron al copiar y pegar: esos `<p>`, esos `&nbsp;`, esos `&amp;` que ensucian todo.

Limpiar UN archivo a mano es tedioso pero se aguanta. ¿Y si son veinte? ¿Y si son doscientos? Ahí la cosa se vuelve imposible a mano. Y el riesgo no es solo el tiempo: si limpiás cien archivos a mano, vas a limpiar cada uno con un criterio un poco distinto. **La máquina, en cambio, aplica SIEMPRE el mismo criterio.** Eso es reproducibilidad, y la reproducibilidad es la base de que tu trabajo se pueda revisar.

Hoy vamos a recorrer una carpeta de archivos sucios (`../datos/crudos/`), limpiarlos en lote, renombrarlos a un patrón consistente (`nota_01.txt`, `nota_02.txt`, …) y guardarlos prolijos en una carpeta nueva. Los originales no se tocan: vos dirigís, la herramienta ejecuta, y siempre te queda el rastro para volver atrás.

> **Ojo:** el corpus es sintético, inventado para el curso, inspirado en la prensa y la conflictividad obrera del Río de la Plata. Sirve para aprender el método, no para citar como fuente.

## Lo que hay en la carpeta sucia

Antes de automatizar nada, miremos qué tenemos. Es la primera regla: no le pidas a la IA que limpie algo que vos no miraste primero.

In [ ]:
# pathlib nos da una forma clara y prolija de trabajar con rutas y carpetas
from pathlib import Path

# Le decimos a Python dónde están los archivos crudos (ruta relativa a esta carpeta)
carpeta_crudos = Path("../datos/crudos")

# glob('*.txt') lista solo los archivos .txt; sorted() los ordena por nombre
archivos = sorted(carpeta_crudos.glob("*.txt"))

# Veamos los nombres tal cual están: desprolijos, con espacios y mayúsculas
for a in archivos:
    print(a.name)

Fijate los nombres: `Nota 1 (copia).txt`, `NOTA_2.txt`, `nota3 FINAL.txt`. Mayúsculas, espacios, paréntesis, numeración inconsistente. Un caos típico de una carpeta real.

## El prompt

Esto es lo que le pediríamos a la IA, en lenguaje natural, como si le explicáramos la tarea a un colega que sabe programar:

> "Tengo una carpeta con varios archivos `.txt` de notas de prensa que están sucios. Necesito un script en Python que recorra TODOS los `.txt` de la carpeta `../datos/crudos/`, que a cada uno le saque los restos de HTML (las etiquetas como `<p>` y las entidades como `&nbsp;` y `&amp;`), que colapse los espacios de más en uno solo y que elimine las líneas vacías que sobran. Que guarde los resultados limpios en una carpeta nueva `./salida/`, renombrados con el patrón `nota_01.txt`, `nota_02.txt`, etcétera. **Importante: no quiero que pise los archivos originales.** Comentá el código en español."

## El código que la IA nos devolvió

Lo primero: una función que recibe un texto sucio y devuelve uno limpio. La separamos del resto porque así el criterio de limpieza queda en UN solo lugar, clarito y revisable.

In [ ]:
# re es el módulo de expresiones regulares: nos deja buscar y reemplazar patrones
import re

# Esta función concentra TODO el criterio de limpieza.
# Recibe el texto sucio (un string) y devuelve el texto ya prolijo.
def limpiar_texto(texto):
    # 1) Sacamos las etiquetas HTML: todo lo que va entre < y >
    texto = re.sub(r"<[^>]+>", " ", texto)

    # 2) Reemplazamos las entidades HTML más comunes por su equivalente
    texto = texto.replace("&nbsp;", " ")  # espacio
    texto = texto.replace("&amp;", "&")   # ampersand
    texto = texto.replace("&lt;", "<")
    texto = texto.replace("&gt;", ">")

    # 3) Colapsamos cualquier corrida de espacios o tabs en un solo espacio
    texto = re.sub(r"[ \t]+", " ", texto)

    # 4) Limpiamos línea por línea: sacamos espacios de los bordes
    #    y descartamos las líneas que quedan vacías
    lineas = [linea.strip() for linea in texto.splitlines()]
    lineas = [linea for linea in lineas if linea != ""]

    # 5) Volvemos a unir con un solo salto de línea entre líneas con contenido
    return "\n".join(lineas)

Ahora el script que recorre la carpeta, aplica la función a cada archivo y guarda el resultado renombrado. Acá está el corazón de la automatización: el bucle que hace de una lo que harías a mano archivo por archivo.

In [ ]:
# Definimos la carpeta de salida (relativa a esta carpeta de la sesión).
# OJO: esta carpeta NO existe todavía; la creamos desde el código al correr.
# Por eso no se versiona: es un resultado, no una fuente. (Va en .gitignore.)
carpeta_salida = Path("./salida")

# mkdir crea la carpeta; exist_ok=True evita el error si ya existía
carpeta_salida.mkdir(exist_ok=True)

# Recorremos cada archivo con su índice. enumerate(..., start=1) arranca en 1.
for i, archivo in enumerate(archivos, start=1):

    # Leemos el archivo crudo completo, como texto UTF-8
    texto_crudo = archivo.read_text(encoding="utf-8")

    # Aplicamos nuestra función de limpieza
    texto_limpio = limpiar_texto(texto_crudo)

    # Armamos el nombre nuevo con el patrón nota_01, nota_02, ...
    # f"{i:02d}" rellena con cero a la izquierda: 1 -> '01'
    nombre_nuevo = f"nota_{i:02d}.txt"
    ruta_destino = carpeta_salida / nombre_nuevo

    # Escribimos el texto limpio en la carpeta de salida.
    # Como escribimos en OTRA carpeta, los originales quedan intactos.
    ruta_destino.write_text(texto_limpio, encoding="utf-8")

    # Avisamos por consola qué pasó con cada archivo (rastro de lo hecho)
    print(f"Limpié {archivo.name} -> {nombre_nuevo}")

## Qué hace y cómo lo validamos

El script hizo, en segundos, lo que vos harías abriendo cada archivo, borrando etiquetas y guardando con otro nombre. Pero **automatizar no es confiar a ciegas.** Validar es parte del trabajo, no un extra. Veamos qué quedó.

In [ ]:
# Listamos lo que se generó en la carpeta de salida
for a in sorted(carpeta_salida.glob("*.txt")):
    print(a.name)

Tres archivos con nombres consistentes. Ahora abrimos uno para ver el contenido limpio:

In [ ]:
# Leemos el primer archivo limpio y lo mostramos
print((carpeta_salida / "nota_01.txt").read_text(encoding="utf-8"))

Y comparemos con el original para confirmar que la limpieza hizo lo que queríamos: que ya no haya `<p>`, ni `&nbsp;`, ni `&amp;`, ni espacios de más.

In [ ]:
# Comparación lado a lado: contamos cuántos restos de HTML quedan
# en los originales vs. en los limpios. Si limpió bien, los limpios dan 0.
def contar_basura(ruta):
    txt = ruta.read_text(encoding="utf-8")
    # Sumamos cuántas etiquetas y entidades HTML aparecen
    return len(re.findall(r"<[^>]+>|&nbsp;|&amp;", txt))

print("Restos de HTML en los ORIGINALES:")
for a in archivos:
    print(f"  {a.name}: {contar_basura(a)}")

print("\nRestos de HTML en los LIMPIOS:")
for a in sorted(carpeta_salida.glob("*.txt")):
    print(f"  {a.name}: {contar_basura(a)}")

Si los originales tienen números mayores a cero y los limpios dan todos cero, la limpieza funcionó. Esa comparación es tu validación: no creés que limpió, lo **comprobás**. Y como los originales siguen ahí con su basura, sabés que no los tocamos.

## Opcional: juntar todo en un solo CSV

A veces no querés tres archivos sueltos sino una tabla con todo junto, lista para analizar en la próxima sesión. Pasemos el corpus limpio a un único CSV con una fila por nota.

In [ ]:
# csv es parte de la biblioteca estándar: no hay que instalar nada
import csv

# Listamos los archivos limpios
limpios = sorted(carpeta_salida.glob("*.txt"))

# Armamos una fila por archivo: su id (el nombre) y su texto completo en una línea
filas = []
for ruta in limpios:
    texto = ruta.read_text(encoding="utf-8").replace("\n", " ")
    filas.append({"id": ruta.name, "texto": texto})

# Escribimos el CSV en la carpeta de salida, con encabezado id,texto
ruta_csv = carpeta_salida / "corpus_limpio.csv"
with ruta_csv.open("w", encoding="utf-8", newline="") as f:
    escritor = csv.DictWriter(f, fieldnames=["id", "texto"])
    escritor.writeheader()
    escritor.writerows(filas)

# Confirmamos mostrando lo que guardamos
for fila in filas:
    print(fila["id"], "->", fila["texto"][:60], "...")

Ahora tenés una tabla limpia, con un identificador prolijo por nota y el texto listo para contar palabras, buscar términos o lo que venga. Pasamos de una carpeta caótica a un dataset ordenado, y todo el proceso quedó escrito y repetible.

## Para jugar

Probá pedirle a la IA estas variaciones y mirá cómo cambia el resultado:

1. **"Agregá una columna `largo` al CSV con la cantidad de caracteres de cada nota, y ordená la tabla de la más larga a la más corta."** Te sirve para ver de un vistazo qué notas tienen más texto.

2. **"Antes de guardar, hacé que el script avise si alguna nota quedó vacía después de limpiar (por ejemplo, si era pura basura HTML)."** Esto te protege de un error silencioso: que un archivo se 'limpie' hasta quedar en nada y vos no te enteres.